# ⚖️ NB-19: DPO – Direct Preference Optimization on Claude Data
**Goal:** Use thinking depth as preference signal to train DPO, aligning a small model to prefer Claude-style thorough reasoning.

**Modality:** Preference Learning | **Model:** TinyLlama + DPO

In [ ]:
!pip install trl peft transformers datasets -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from trl import DPOTrainer, DPOConfig
from peft import LoraConfig, get_peft_model, TaskType
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset
import torch, random

random.seed(42)
MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tok = AutoTokenizer.from_pretrained(MODEL); tok.pad_token = tok.eos_token
base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
model = get_peft_model(base, LoraConfig(task_type=TaskType.CAUSAL_LM, r=4, lora_alpha=16,
    target_modules=["q_proj","v_proj"], lora_dropout=0.05))

# Build preference dataset: chosen = Claude full response, rejected = truncated/shallow
dpo_data = []
for d in data:
    if len(d["thinking"]) > 100 and len(d["response"]) > 100:
        chosen  = d["response"]  # full thoughtful response
        rejected = d["response"][:50] + "... [truncated, shallow answer]"  # simulated bad response
        dpo_data.append({"prompt": d["user"], "chosen": chosen[:512], "rejected": rejected})

print(f"DPO pairs: {len(dpo_data)}")
ds = Dataset.from_list(dpo_data).train_test_split(0.1)

cfg = DPOConfig(
    output_dir="./dpo-claude",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    beta=0.1,  # DPO temperature
    max_length=512,
    max_prompt_length=128,
    fp16=torch.cuda.is_available(),
    report_to="none"
)
DPOTrainer(model=model, ref_model=None, args=cfg,
           train_dataset=ds["train"], eval_dataset=ds["test"], tokenizer=tok).train()
print("✅ DPO alignment complete!")
